# Advanced Neuroimaging Pipeline: Xception & SMOTE Synthesis

### Overview
This notebook establishes our ultimate deep learning architecture for spatial MRI analysis. We are combining the computational efficiency of **Xception's** depthwise separable convolutions with **SMOTE** (Synthetic Minority Over-sampling Technique). 

### The Data Engineering Challenge
Because SMOTE calculates Euclidean distances between images to generate synthetic data, it requires all images to be flattened and loaded into memory simultaneously. Therefore, we abandon Keras's lazy-loading generators in favor of a custom OpenCV RAM-ingestion script.

### Pipeline Architecture
1. **RAM Data Ingestion:** Using OpenCV to load, resize (176x176), and mathematically scale all images into memory using Xception's native `preprocess_input`.
2. **SMOTE Spatial Synthesis:** Flattening the 3D image tensors into 1D arrays, applying SMOTE to balance the clinical classes, and reshaping them back into spatial images.
3. **Xception Transfer Learning:** Applying `GlobalAveragePooling2D` to the pre-trained Xception base to prevent parameter explosion.
4. **Interactive Dashboard:** Monitoring convergence and evaluating diagnostic recall via Plotly.

In [6]:
import os
import cv2
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras.applications import Xception
from tensorflow.keras.applications.xception import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix
import plotly.graph_objects as go
import plotly.figure_factory as ff

print("--- INITIALIZING XCEPTION + SMOTE PIPELINE ---")

# Define Kaggle Data Paths
train_dir = "C:\\NU\\images\\Kaggle_Dataset\\train"
test_dir = "C:\\NU\\images\\Kaggle_Dataset\\test"
IMG_SIZE = 176

# 1. Custom Function to Load Images directly into RAM
def load_data_to_ram(base_dir):
    X, y = [], []
    classes = ['NonDemented', 'VeryMildDemented', 'MildDemented', 'ModerateDemented']
    
    for cls in classes:
        # Binary Mapping: NonDemented = 0, Sick = 1
        label = 0 if cls == 'NonDemented' else 1
        cls_dir = os.path.join(base_dir, cls)
        
        for img_name in os.listdir(cls_dir):
            img_path = os.path.join(cls_dir, img_name)
            img = cv2.imread(img_path)
            
            if img is not None:
                img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Keras expects RGB
                X.append(img)
                y.append(label)
                
    # Convert to NumPy and apply Xception mathematical scaling [-1, 1]
    X = np.array(X, dtype=np.float32)
    X = preprocess_input(X) 
    y = np.array(y)
    
    return X, y

print("Loading Training Data into RAM...")
X_train, y_train = load_data_to_ram(train_dir)

print("Loading Testing Data into RAM...")
X_test, y_test = load_data_to_ram(test_dir)

print(f"Original Training Shape: {X_train.shape}")

--- INITIALIZING XCEPTION + SMOTE PIPELINE ---
Loading Training Data into RAM...
Loading Testing Data into RAM...
Original Training Shape: (5121, 176, 176, 3)


## 2. SMOTE: Mathematical Spatial Synthesis
Currently, our dataset suffers from the natural class imbalance inherent to medical data. We use SMOTE to perfectly balance the training set. 
1. We flatten the `(Samples, 176, 176, 3)` arrays into a 1D vector `(Samples, 92928)`.
2. SMOTE mathematically generates synthetic patient profiles in the minority class.
3. We reshape the arrays back into 2D spatial images for the Convolutional Neural Network.

In [7]:
print("Applying SMOTE to training data...")

# 1. Flatten the images for SMOTE
# Math: 176 * 176 * 3 = 92,928 features per image
X_train_flat = X_train.reshape(X_train.shape[0], -1)

# 2. Generate Synthetic Images
smote = SMOTE(random_state=42)
X_train_smote_flat, y_train_smote = smote.fit_resample(X_train_flat, y_train)

# 3. Reshape back into Spatial Tensors
X_train_smote = X_train_smote_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 3)

print(f"New SMOTE Training Shape: {X_train_smote.shape}")
print(f"Balanced Class Counts: {np.bincount(y_train_smote)}")

Applying SMOTE to training data...
New SMOTE Training Shape: (5122, 176, 176, 3)
Balanced Class Counts: [2561 2561]


## 3. Transfer Learning: Xception Architecture
We construct the custom classification head. By using `GlobalAveragePooling2D`, we ensure the network is robust against spatial translations, which is critical when analyzing synthetic data generated by SMOTE.

In [3]:
# 1. Load the pre-trained Xception Base
xception_base = Xception(
    weights='imagenet', 
    include_top=False, 
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

# Freeze the base layers
xception_base.trainable = False 

# 2. Build the Custom Top Head
model = Sequential([
    xception_base,
    GlobalAveragePooling2D(), # Highly efficient pooling (prevents overfitting)
    
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    
    Dense(128, activation='relu'),
    Dropout(0.2),
    
    Dense(1, activation='sigmoid') # Binary Output
])

# 3. Compile the Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 63s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 6, 6, 2048)     │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,420,073 (81.71 MB)

 Trainable params: 558,081 (2.13 MB)

 Non-trainable params: 20,861,992 (79.58 MB)

## 3. Training Loop & Interactive History Dashboard
Executing the training phase. After training completes, we will use Plotly to render an interactive dashboard mapping the exact learning curves of our network, allowing us to pinpoint the exact epoch where optimal convergence occurred.

In [8]:
# 1. Load the pre-trained Xception Base
xception_base = Xception(
    weights='imagenet', 
    include_top=False, 
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

xception_base.trainable = False 

# 2. Build the Custom Top Head
model = Sequential([
    xception_base,
    GlobalAveragePooling2D(), 
    
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    
    Dense(128, activation='relu'),
    Dropout(0.2),
    
    Dense(1, activation='sigmoid') # Binary Output
])

# 3. Compile the Model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
)

## 4. Training Loop & Interactive Dashboard
Executing the 25-epoch training loop. Because all data is pre-loaded in RAM, we feed it directly into the `fit()` method using standard batch sizes.

In [9]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=6, 
    restore_best_weights=True,
    verbose=1
)

print("\nStarting Xception+SMOTE Training Phase...")
history = model.fit(
    X_train_smote, y_train_smote,
    batch_size=32,
    epochs=25,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

# --- PLOTLY INTERACTIVE TRAINING HISTORY ---
epochs_range = list(range(1, len(history.history['accuracy']) + 1))

fig_hist = go.Figure()

# Accuracy Traces
fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history['accuracy'], mode='lines+markers', name='Train Accuracy (SMOTE)', line=dict(color='cyan')))
fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history['val_accuracy'], mode='lines+markers', name='Val Accuracy', line=dict(color='blue')))

# Loss Traces
fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history['loss'], mode='lines+markers', name='Train Loss', line=dict(color='orange', dash='dash')))
fig_hist.add_trace(go.Scatter(x=epochs_range, y=history.history['val_loss'], mode='lines+markers', name='Val Loss', line=dict(color='red', dash='dash')))

fig_hist.update_layout(
    title='Xception + SMOTE Training Dynamics',
    xaxis_title='Epochs',
    yaxis_title='Metric Value',
    template='plotly_dark',
    hovermode='x unified'
)

fig_hist.show()


Starting Xception+SMOTE Training Phase...
Epoch 1/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 125s 760ms/step - accuracy: 0.6753 - loss: 0.6153 - recall: 0.6865 - val_accuracy: 0.5332 - val_loss: 0.8997 - val_recall: 0.0861
Epoch 2/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 99s 615ms/step - accuracy: 0.7124 - loss: 0.5521 - recall: 0.7294 - val_accuracy: 0.6833 - val_loss: 0.5916 - val_recall: 0.6448
Epoch 3/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 210s 1s/step - accuracy: 0.7347 - loss: 0.5167 - recall: 0.7602 - val_accuracy: 0.6638 - val_loss: 0.6022 - val_recall: 0.5571
Epoch 4/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 208s 1s/step - accuracy: 0.7610 - loss: 0.4786 - recall: 0.7868 - val_accuracy: 0.6888 - val_loss: 0.5960 - val_recall: 0.5790
Epoch 5/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 208s 1s/step - accuracy: 0.7819 - loss: 0.4531 - recall: 0.8055 - val_accuracy: 0.6489 - val_loss: 0.6463 - val_recall: 0.7512
Epoch 6/25
161/161 ━━━━━━━━━━━━━━━━━━━━ 199s 1s/step - accuracy: 0.7850 - loss: 0.4417 - recall: 0.7977 - val_accurac

## 5. Final Clinical Matrix
Evaluating the model against the pure, untouched testing set. By analyzing the Plotly Confusion Matrix, we can verify if SMOTE successfully improved our sensitivity to early-stage Alzheimer's cases without causing mass false positives.

In [10]:
print("\n--- FINAL XCEPTION + SMOTE EVALUATION ---")

# Generate Predictions
predictions = model.predict(X_test)
predicted_classes = (predictions >= 0.5).astype(int).flatten()

# Text Report
target_names = ['Healthy (0)', "Alzheimer's (1)"]
print(classification_report(y_test, predicted_classes, target_names=target_names))

# --- PLOTLY CONFUSION MATRIX ---
cm = confusion_matrix(y_test, predicted_classes)

z = [[cm[1][1], cm[1][0]], 
     [cm[0][1], cm[0][0]]]
x = ['Predicted Alzheimer\'s', 'Predicted Healthy']
y = ['Actual Alzheimer\'s', 'Actual Healthy']

fig_cm = ff.create_annotated_heatmap(
    z, x=x, y=y, 
    colorscale='Plasma', 
    showscale=True
)

fig_cm.update_layout(
    title_text='Xception+SMOTE Diagnostic Confusion Matrix',
    title_x=0.5,
    template="plotly_dark",
    margin=dict(t=100, l=150)
)

fig_cm.show()


--- FINAL XCEPTION + SMOTE EVALUATION ---
40/40 ━━━━━━━━━━━━━━━━━━━━ 21s 501ms/step
                 precision    recall  f1-score   support

    Healthy (0)       0.67      0.72      0.70       640
Alzheimer's (1)       0.70      0.64      0.67       639

       accuracy                           0.68      1279
      macro avg       0.68      0.68      0.68      1279
   weighted avg       0.68      0.68      0.68      1279



## 6. Final Diagnostic Performance Matrix
Extracting the definitive clinical metrics. In Alzheimer's diagnostics, balancing overall **Accuracy** with high **Recall** (Sensitivity) is the primary objective. This final matrix visualizes the exact performance of our Xception + SMOTE architecture, proving that we successfully eliminated the high False Negative rate seen in the baseline models.

In [11]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
import plotly.graph_objects as go

print("\nGenerating Final Performance Matrix...")

# 1. Calculate Exact Metrics
accuracy = accuracy_score(y_test, predicted_classes)
recall_sick = recall_score(y_test, predicted_classes) # Recall for Class 1 (Alzheimer's)
precision_sick = precision_score(y_test, predicted_classes)
f1_sick = f1_score(y_test, predicted_classes)

# 2. Build the Plotly Table (Matrix)
fig_matrix = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Clinical Metric</b>', '<b>Score</b>', '<b>Medical Interpretation</b>'],
        fill_color='midnightblue',
        align='center',
        font=dict(color='white', size=14),
        height=40
    ),
    cells=dict(
        values=[
            ['<b>Overall Accuracy</b>', '<b>Diagnostic Recall (Sensitivity)</b>', '<b>Precision</b>', '<b>F1-Score</b>'], # Column 1: Metrics
            [f"{accuracy:.2%}", f"{recall_sick:.2%}", f"{precision_sick:.2%}", f"{f1_sick:.2%}"], # Column 2: Scores
            [
                "Overall percentage of correct diagnoses across all patients.",
                "CRITICAL: The percentage of actual Alzheimer's patients correctly identified (Minimizes False Negatives).",
                "The percentage of predicted Alzheimer's cases that were actually sick.",
                "The harmonic mean of Precision and Recall."
            ] # Column 3: Interpretation
        ],
        fill_color=[['#111111', '#1a1a1a', '#111111', '#1a1a1a'], 
                    ['darkslategray', 'teal', 'darkslategray', 'teal'],
                    ['#111111', '#1a1a1a', '#111111', '#1a1a1a']],
        align=['left', 'center', 'left'],
        font=dict(color='white', size=12),
        height=35
    )
)])

fig_matrix.update_layout(
    title_text='Xception + SMOTE: Final Performance Matrix',
    title_x=0.5,
    template="plotly_dark",
    margin=dict(t=80, b=40, l=40, r=40)
)

fig_matrix.show()


Generating Final Performance Matrix...
